In [6]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/Malay-English Code-Switched Medical Query Dataset Collection Study2.xlsx")

symptom_columns = [
'Fever',
'Emergency',
'Respiratory',
'Digestive',
'Neurological'
]

# Drop duplicate
df = df.drop_duplicates()

# Remove useless responses
noise_patterns = [
    "doesnt make sense",
    "doesn't make sense",
    "too hard to answer",
    "what kind are these questions"
]

for col in df.columns:

    if df[col].dtype == 'object':

        for pattern in noise_patterns:

            df = df[
                ~df[col]
                .astype(str)
                .str.lower()
                .str.contains(pattern, na=False)
            ]

# Stopwords
custom_stopwords = {
    'i', 'my', 'is', 'am', 'the', 'and',
    'have', 'very', 'saya', 'rasa', 'lah', 'ni',
    'itu', 'dan', 'yang'
}

def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word for word in words
        if word not in custom_stopwords
    ]

    return ' '.join(filtered_words)

# Lemmatization
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):

    words = text.split()

    lemmatized_words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return ' '.join(lemmatized_words)

# Phrase

phrase_translation = {

    'sakit kepala': 'headache',
    'sesak nafas': 'shortness_of_breath',
    'chest pain': 'chest_pain',
    'sore throat': 'sore_throat',
    'shortness of breath': 'shortness_of_breath'
}

def normalize_phrase_terms(text):

    for phrase, replacement in phrase_translation.items():

        text = text.replace(phrase, replacement)

    return text

medical_translation = {

    # Malay → English
    'demam': 'fever',
    'batuk': 'cough',
    'sakit kepala': 'headache',
    'pening': 'dizziness',
    'sesak nafas': 'shortness_of_breath',
    'selsema': 'flu',
    'selesema': 'flu',
    'kahak': 'phlegm',
    'muntah': 'vomit',
    'cirit birit': 'diarrhea',
    'pengsan': 'faint',

    # Manglish
    'tak': 'not',
    'x': 'not',
    'ade': 'ada',
    'doc': 'doctor',

    # spelling normalization
    'phlegms': 'phlegm',
    'feverish': 'fever',
    'dizzy': 'dizziness'
}

# Create phrase normalization function

def normalize_medical_terms(text):

    for phrase, replacement in medical_translation.items():

        text = text.replace(phrase, replacement)

    return text

spelling_dict = {

    # Malay spelling variants
    'selsema': 'selesema',
    'demamgila': 'demam teruk',

    # English spelling variants
    'phlegms': 'phlegm',
    'feverish': 'fever',
    'dizzy': 'dizziness',
    'throbbing': 'throb',
    'aching': 'ache',
    'coughing': 'cough',

    # informal spelling
    'soo': 'so',
    'alot': 'a lot',

    # typo examples
    'temprature': 'temperature',
    'brething': 'breathing'
}

def normalize_spelling(text):

    words = text.split()

    normalized_words = [

        spelling_dict.get(word, word)

        for word in words
    ]

    return ' '.join(normalized_words)

import re

def preprocess(text):

    # handle missing values
    if pd.isna(text):
        return ''

    if isinstance(text, str):

        # lowercase
        text = text.lower()

        # normalize unicode punctuation
        text = text.replace('’', "'")
        text = text.replace('‘', "'")
        text = text.replace('“', '"')
        text = text.replace('”', '"')
        text = text.replace('，', ',')
        text = text.replace('。', '.')

        # remove encoding garbage
        text = re.sub(r'[�ÃÂâ€™ï»¿]', '', text)

        # remove emoji/symbols
        text = re.sub(r'[^\w\s.,!?_]','', text)

        # normalized repeated letters
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)

        # remove mixed punctuation
        text = re.sub(r'([!?.,])\1+', r'\1', text)

        # remove extra spaces
        text = ' '.join(text.split())

        # remove edge punctuation
        text = text.strip(".,!? ")

        # phrase normalization
        text = normalize_phrase_terms(text)

        # spelling normalization
        text = normalize_spelling(text)

        # temperature & medical normalization
        text = re.sub(r'\b\d+(\.\d+)?\b',' temperature ',text)
        text = normalize_medical_terms(text)

        # remove stopwords
        text = remove_stopwords(text)

        # lemmatization
        text = lemmatize_text(text)

        # final whitespace cleanup
        text = ' '.join(text.split())

    return text

for col in symptom_columns:
    df[col] = df[col].apply(preprocess)

# remove empty rows
for col in symptom_columns:

    df = df[
        df[col].str.strip() != ''
    ]

# =========================
# DATA ANNOTATION
# =========================

annotated_df = pd.melt(

    df,

    value_vars=symptom_columns,

    var_name='label',

    value_name='text'
)

# remove missing values
annotated_df = annotated_df.dropna()

# remove empty text
annotated_df = annotated_df[
    annotated_df['text'].str.strip() != ''
]

# reset index
annotated_df = annotated_df.reset_index(drop=True)

# check annotation output
print(annotated_df.head())

# check class distribution
print(annotated_df['label'].value_counts())

# save annotated dataset
annotated_df.to_csv(
    'annotated_medical_dataset.csv',
    index=False
)

#----------------------
# Split train/test set
#----------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    annotated_df['text'],
    annotated_df['label'],

    test_size=0.2,

    stratify=annotated_df['label'],

    random_state=42
)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   label                                               text
0  Fever  yesterday night feeling hot headache, use ther...
1  Fever                      a high temperature feel tired
2  Fever                                              fever
3  Fever                                              fever
4  Fever                   a fever body feel like hot tired
label
Fever           299
Emergency       299
Respiratory     299
Digestive       299
Neurological    299
Name: count, dtype: int64


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [7]:
print(len(X_train))
print(len(X_test))

1196
299
